# PrimeNet on MIMIC-IV (Colab)

Train **neutropenic fever (NF)** with vendored PrimeNet in ChemoTreeVsDL (same folds / top-100 labs as STraTS).

| Script | Role |
|--------|------|
| [`colab_primenet_train.py`](../colab_primenet_train.py) | Prepare data + train |
| [`docs/PRIMENET.md`](../docs/PRIMENET.md) | Integration notes |

**Runtime:** GPU (T4 is enough for `--fast` smoke test).

**Data:** upload CSVs to `data/raw/`, or copy `MIMIC_IV/saved_data/` from Drive.

## 1. Clone repo and install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = "bionetslab/ChemoTreeVsDL"  # or your fork
BRANCH = "feature/primenet-vendored"

if not Path("ChemoTreeVsDL").is_dir():
    subprocess.check_call(
        ["git", "clone", "-b", BRANCH, f"https://github.com/{REPO}.git"]
    )
os.chdir("ChemoTreeVsDL")

root = Path.cwd()
sys.path.insert(0, str(root))
os.environ["PYTHONPATH"] = str(root)
os.environ["PRIMENET_ROOT"] = str((root / "third_party" / "PrimeNet").resolve())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
print("Ready:", root)

## 2. Options

In [ ]:
RUN_FAST = True
RUN_ALL_FOLDS = False
FOLD = 0
PREFIX = "colab_primenet"
SKIP_PREPARE = False

USE_DRIVE_SAVED_DATA = False
DRIVE_SAVED_DATA = "/content/drive/MyDrive/ChemoTreeVsDL/MIMIC_IV/saved_data"

SAVE_TO_DRIVE = False
DRIVE_OUT = "/content/drive/MyDrive/ChemoTreeVsDL/MIMIC_IV/saved_data"

## 3. Run pipeline

In [ ]:
if USE_DRIVE_SAVED_DATA:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    dst = Path("MIMIC_IV/saved_data")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_SAVED_DATA, dst, dirs_exist_ok=True)
    SKIP_PREPARE = True
    print("Copied saved_data from Drive")

cmd = [sys.executable, "colab_primenet_train.py", "--prefix", PREFIX]
if RUN_FAST:
    cmd.append("--fast")
if SKIP_PREPARE:
    cmd.append("--skip-prepare")
if RUN_ALL_FOLDS:
    cmd.append("--all-folds")
else:
    cmd.extend(["--fold", str(FOLD)])

subprocess.check_call(cmd)

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil

    drive.mount("/content/drive")
    shutil.copytree("MIMIC_IV/saved_data", DRIVE_OUT, dirs_exist_ok=True)
    print("Copied results to Drive")

## 4. Summarize folds (optional)

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "scripts/summarize_folds.py",
        "--prefix",
        PREFIX,
        "--model",
        "primenet",
    ]
)